# Approach 1: SVM + TF-IDF (Non-neural Baseline)

Multi-label intent classification on informal Vietnamese customer chat data.

**Model:** One-vs-Rest `LinearSVC` with combined word + character TF-IDF features  
**Purpose:** Non-neural lower bound — establishes how much complexity is needed before moving to BERT-based approaches.

**Prerequisite:** Run `data_preparation/data_split.ipynb` first to generate the shared split files in `final_data/`.

| Metric | Role |
|---|---|
| Macro-F1 | Primary — weights all 32 classes equally |
| Micro-F1 | Label-level, weighted by frequency |
| Hamming Loss | Fraction of individual label predictions that are wrong |
| Subset Accuracy | Full label set exactly correct |

In [ ]:
# Requirements for this notebook
# See requirements.txt at the project root for the full list
!pip install scikit-learn scikit-multilearn scipy numpy pandas matplotlib seaborn joblib --quiet

In [ ]:
import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack, vstack as sp_vstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score, classification_report
)

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)

DATA_DIR = '../final_data'

## 1. Load Pre-split Data

Loads the train / val / test JSONL files and the shared `MultiLabelBinarizer` produced by `data_preparation/data_split.ipynb`.

In [ ]:
def load_jsonl(path):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            texts.append(item['text'])
            labels.append(item['cats'])
    return texts, labels

train_texts, train_labels = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_texts,   val_labels   = load_jsonl(f'{DATA_DIR}/val.jsonl')
test_texts,  test_labels  = load_jsonl(f'{DATA_DIR}/test.jsonl')

mlb = joblib.load(f'{DATA_DIR}/mlb.joblib')

y_train = mlb.transform(train_labels)
y_val   = mlb.transform(val_labels)
y_test  = mlb.transform(test_labels)

print(f"Train : {len(train_texts):4d} samples")
print(f"Val   : {len(val_texts):4d} samples")
print(f"Test  : {len(test_texts):4d} samples")
print(f"Labels: {len(mlb.classes_)} classes")

## 2. TF-IDF Feature Extraction

Two vectorizers are combined:

| Vectorizer | N-gram range | Max features | Purpose |
|---|---|---|---|
| Word TF-IDF | (1, 2) | 30,000 | Word-level and short phrase patterns |
| Char TF-IDF | (2, 4) | 20,000 | Captures teen code substrings (`"kh"`, `"ko"`, `"đc"`) that word tokenization misses |

Both vectorizers are **fit on training data only**, then applied to val and test.

In [ ]:
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=30_000,
    min_df=2,
    sublinear_tf=True  # log(1+tf) — reduces impact of high-frequency terms
)

tfidf_char = TfidfVectorizer(
    analyzer='char_wb',  # pads word boundaries, avoids cross-word n-grams
    ngram_range=(2, 4),
    max_features=20_000,
    min_df=2,
    sublinear_tf=True
)

X_train_word = tfidf_word.fit_transform(train_texts)
X_train_char = tfidf_char.fit_transform(train_texts)

X_val_word   = tfidf_word.transform(val_texts)
X_val_char   = tfidf_char.transform(val_texts)

X_test_word  = tfidf_word.transform(test_texts)
X_test_char  = tfidf_char.transform(test_texts)

X_train_feat = hstack([X_train_word, X_train_char])
X_val_feat   = hstack([X_val_word,   X_val_char])
X_test_feat  = hstack([X_test_word,  X_test_char])

print(f"Feature matrix — train: {X_train_feat.shape}")
print(f"  Word features : {X_train_word.shape[1]:,}")
print(f"  Char features : {X_train_char.shape[1]:,}")

## 3. Train and Tune SVM

`OneVsRestClassifier` trains one `LinearSVC` per intent class (32 binary classifiers total).  
Regularization strength `C` is tuned on validation Macro-F1 across `{0.1, 1.0, 10.0}`.  
`class_weight='balanced'` compensates for the label frequency imbalance.

In [ ]:
best_c, best_val_macro_f1 = None, 0.0
tuning_log = []

for C in [0.1, 1.0, 10.0]:
    clf = OneVsRestClassifier(
        LinearSVC(C=C, class_weight='balanced', max_iter=2000),
        n_jobs=-1
    )
    clf.fit(X_train_feat, y_train)
    y_pred_val = clf.predict(X_val_feat)

    macro_f1 = f1_score(y_val, y_pred_val, average='macro', zero_division=0)
    micro_f1 = f1_score(y_val, y_pred_val, average='micro', zero_division=0)
    tuning_log.append({'C': C, 'Val Macro-F1': macro_f1, 'Val Micro-F1': micro_f1})
    print(f"  C={C:5.1f}  |  Val Macro-F1: {macro_f1:.4f}  |  Val Micro-F1: {micro_f1:.4f}")

    if macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = macro_f1
        best_c = C

print(f"\nBest C = {best_c}  (Val Macro-F1 = {best_val_macro_f1:.4f})")

In [ ]:
# Retrain on train + val combined with best C before final evaluation
X_trainval_feat = sp_vstack([X_train_feat, X_val_feat])
y_trainval = np.vstack([y_train, y_val])

final_model = OneVsRestClassifier(
    LinearSVC(C=best_c, class_weight='balanced', max_iter=2000),
    n_jobs=-1
)
final_model.fit(X_trainval_feat, y_trainval)
print(f"Final model trained on train + val ({len(train_texts) + len(val_texts)} samples).")

## 4. Evaluate on Test Set

In [ ]:
y_pred_test = final_model.predict(X_test_feat)

macro_f1   = f1_score(y_test, y_pred_test, average='macro',  zero_division=0)
micro_f1   = f1_score(y_test, y_pred_test, average='micro',  zero_division=0)
h_loss     = hamming_loss(y_test, y_pred_test)
subset_acc = accuracy_score(y_test, y_pred_test)

print("=" * 45)
print("TEST RESULTS — Approach 1: SVM + TF-IDF")
print("=" * 45)
print(f"Macro-F1        : {macro_f1:.4f}")
print(f"Micro-F1        : {micro_f1:.4f}")
print(f"Hamming Loss    : {h_loss:.4f}")
print(f"Subset Accuracy : {subset_acc:.4f}")

In [ ]:
# Per-class classification report
print(classification_report(
    y_test, y_pred_test,
    target_names=mlb.classes_,
    zero_division=0
))

In [ ]:
# Per-class F1 bar chart
per_class_f1 = f1_score(y_test, y_pred_test, average=None, zero_division=0)
df_f1 = pd.DataFrame({'Label': mlb.classes_, 'F1': per_class_f1})\
           .sort_values('F1', ascending=True)

colors = ['#d62728' if f < 0.5 else '#ff7f0e' if f < 0.7 else '#2ca02c'
          for f in df_f1['F1']]

plt.figure(figsize=(12, 10))
plt.barh(df_f1['Label'], df_f1['F1'], color=colors)
plt.axvline(macro_f1, color='steelblue', linestyle='--', linewidth=1.5,
            label=f'Macro-F1 = {macro_f1:.3f}')
plt.xlabel('F1 Score')
plt.title('Per-class F1 — Approach 1: SVM + TF-IDF')
plt.legend()
plt.tight_layout()
plt.savefig('results/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Lowest F1 : {df_f1.iloc[0]['Label']}  ({df_f1.iloc[0]['F1']:.3f})")
print(f"Highest F1: {df_f1.iloc[-1]['Label']} ({df_f1.iloc[-1]['F1']:.3f})")

## 4b. Learning Curve

Shows how Macro-F1 changes as the model sees more training data — the standard diagnostic for non-neural models, equivalent to a loss curve for neural networks.

| Pattern | Meaning |
|---|---|
| Train F1 >> Val F1 (large gap) | Overfitting — model memorises training data |
| Both curves low and flat | Underfitting — need a better model or features |
| Train ≈ Val F1, both high | Well-fitted — more data gives diminishing returns |

In [ ]:
fractions   = np.linspace(0.1, 1.0, 10)
train_f1s   = []
val_f1s     = []
train_sizes = []

n_total = X_train_feat.shape[0]

for frac in fractions:
    n = max(1, int(n_total * frac))
    X_sub = X_train_feat[:n]
    y_sub = y_train[:n]

    clf = OneVsRestClassifier(
        LinearSVC(C=best_c, class_weight='balanced', max_iter=2000),
        n_jobs=-1
    )
    clf.fit(X_sub, y_sub)

    train_pred = clf.predict(X_sub)
    val_pred   = clf.predict(X_val_feat)

    train_f1s.append(f1_score(y_sub, train_pred, average='macro', zero_division=0))
    val_f1s.append(  f1_score(y_val, val_pred,   average='macro', zero_division=0))
    train_sizes.append(n)
    print(f'  {frac*100:5.0f}% ({n:4d} samples) — Train F1: {train_f1s[-1]:.4f}  Val F1: {val_f1s[-1]:.4f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_f1s, 'o-', color='#ee854a', label='Train Macro-F1')
ax.plot(train_sizes, val_f1s,   's-', color='#4878d0', label='Val Macro-F1')
ax.fill_between(train_sizes, train_f1s, val_f1s,
                alpha=0.12, color='gray', label='Gap (overfitting region)')
ax.set_xlabel('Training samples used')
ax.set_ylabel('Macro-F1')
ax.set_title('Learning Curve — Approach 1: SVM + TF-IDF (Baseline)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('results/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFinal gap (Train - Val): {train_f1s[-1] - val_f1s[-1]:+.4f}')

## 5. Save Model and Results

In [ ]:
joblib.dump(final_model, 'results/svm_model.joblib')
joblib.dump(tfidf_word,  'results/tfidf_word.joblib')
joblib.dump(tfidf_char,  'results/tfidf_char.joblib')

metrics = {
    'approach'       : 'SVM + TF-IDF',
    'best_C'         : best_c,
    'macro_f1'       : round(macro_f1,   4),
    'micro_f1'       : round(micro_f1,   4),
    'hamming_loss'   : round(h_loss,     4),
    'subset_accuracy': round(subset_acc, 4),
    'per_class_f1'   : {k: round(float(v), 4) for k, v in zip(mlb.classes_, per_class_f1)}
}

with open('results/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved to approach1/results/:")
for fname in ['svm_model.joblib', 'tfidf_word.joblib', 'tfidf_char.joblib',
              'metrics.json', 'per_class_f1.png']:
    print(f"  {fname}")

## 6. Test Custom Input

Predict intents for any message you type. This section reloads everything from `results/` so it works even after a kernel restart.

In [ ]:
import joblib, json, datetime
import numpy as np
from scipy.sparse import hstack

_model      = joblib.load('results/svm_model.joblib')
_tfidf_word = joblib.load('results/tfidf_word.joblib')
_tfidf_char = joblib.load('results/tfidf_char.joblib')
_mlb        = joblib.load('../final_data/mlb.joblib')

# In-memory log — persists across predict() calls within this session
_prediction_log = []

def predict(text: str, top_k: int = 3):
    feat = hstack([
        _tfidf_word.transform([text]),
        _tfidf_char.transform([text])
    ])

    predicted = _mlb.inverse_transform(_model.predict(feat))[0]

    # Decision function scores — rank all 32 classes by confidence
    scores = _model.decision_function(feat)[0]
    ranked = sorted(zip(_mlb.classes_, scores), key=lambda x: -x[1])

    # Append to history log (LinearSVC has no probabilities, so scores are stored)
    _prediction_log.append({
        'timestamp' : datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'input'     : text,
        'predicted' : list(predicted),
        'top_scores': {label: round(float(score), 4) for label, score in ranked[:top_k]},
    })

    print(f"Input : {text}")
    print(f"Predicted : {list(predicted) if predicted else '[none above threshold]'}")
    print(f"Top {top_k} scores:")
    for label, score in ranked[:top_k]:
        marker = " <--" if label in predicted else ""
        print(f"  {label:<35} {score:+.3f}{marker}")

print("predict() ready.  History log is in _prediction_log.")

In [ ]:
# ── Change this text and re-run ───────────────────────────────────────────────
predict("shop ơi còn hàng con ferrari không ạ")

In [ ]:
# Test multiple messages at once
samples = [
    "shop ơi",
    "bao g có hàng thế ạ",
    "cho e đặt màu xanh với ạ",
    "giá con này bao nhiêu vậy shop",
]

for text in samples:
    print("-" * 55)
    predict(text, top_k=2)
print("-" * 55)

## 7. Prediction History Log

Every `predict()` call is recorded in `_prediction_log` (in-memory) and can be:

- **Viewed** as a table with `show_log()`
- **Saved** to `results/prediction_log.jsonl` with `save_log()`
- **Cleared** with `clear_log()`

Each entry records: timestamp, original input, predicted intents, and top-k decision scores.  
*(Decision scores, not probabilities — LinearSVC does not produce calibrated probabilities.)*

In [ ]:
def show_log(n: int = None):
    """Print the prediction history as a table. n=None shows all entries."""
    entries = _prediction_log if n is None else _prediction_log[-n:]
    if not entries:
        print('Log is empty — run predict() first.')
        return
    print(f'{"#":<4} {"Timestamp":<20} {"Input":<40} {"Predicted"}')
    print('-' * 100)
    for i, e in enumerate(entries):
        idx   = len(_prediction_log) - len(entries) + i + 1
        ts    = e['timestamp']
        inp   = e['input'][:38] + '..' if len(e['input']) > 40 else e['input']
        preds = ', '.join(e['predicted']) if e['predicted'] else '[none]'
        print(f'{idx:<4} {ts:<20} {inp:<40} {preds}')
    print(f'\nTotal entries: {len(_prediction_log)}')

show_log()

In [ ]:
LOG_PATH = 'results/prediction_log.jsonl'

def save_log(path: str = LOG_PATH, mode: str = 'append'):
    """
    Save _prediction_log to a JSONL file.
    mode='append' adds new entries to the end of an existing file.
    mode='overwrite' replaces the file entirely.
    """
    if not _prediction_log:
        print('Nothing to save — log is empty.')
        return

    if mode == 'append':
        existing_keys = set()
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    e = json.loads(line)
                    existing_keys.add((e['timestamp'], e['input']))
        except FileNotFoundError:
            pass

        new_entries = [e for e in _prediction_log
                       if (e['timestamp'], e['input']) not in existing_keys]

        with open(path, 'a', encoding='utf-8') as f:
            for e in new_entries:
                json.dump(e, f, ensure_ascii=False)
                f.write('\n')

        print(f'Appended {len(new_entries)} new entries → {path}')

    else:  # overwrite
        with open(path, 'w', encoding='utf-8') as f:
            for e in _prediction_log:
                json.dump(e, f, ensure_ascii=False)
                f.write('\n')
        print(f'Saved {len(_prediction_log)} entries → {path} (overwrite)')


def clear_log():
    """Clear the in-memory log (does NOT delete the saved JSONL file)."""
    count = len(_prediction_log)
    _prediction_log.clear()
    print(f'Cleared {count} entries from in-memory log.')


# ── Usage examples ────────────────────────────────────────────────────────────
# save_log()                     # append new entries to results/prediction_log.jsonl
# save_log(mode='overwrite')     # replace the file
# show_log(n=5)                  # show last 5 entries
# clear_log()                    # wipe in-memory log
print('save_log() / clear_log() ready.')